# 03 – Sampling and the Central Limit Theorem

You almost never have data on every person, transaction, or item you care about.

You have **a sample** — a subset of a much larger population.

A survey of 1,000 voters tells you about 100 crore voters. A batch test of 50 phones tells you about the entire production run. A month's worth of transactions tells you about the long-term pattern.

This is only useful because of a remarkable mathematical result: the **Central Limit Theorem (CLT)**.

The CLT is the foundation of statistical inference — the process of drawing reliable conclusions about a population from a sample. It is arguably the single most important theorem in applied statistics.

## 1) Populations vs Samples

**Population:** Every member of the group you want to study
- All adults in India
- All transactions in 2023
- All possible outcomes of a manufacturing process

**Sample:** A subset you actually measure
- 2,000 randomly selected adults
- Last month's transactions
- 100 items from today's production run

**The goal:** Use measurements from the sample to make statements about the population.

**Key terms:**
- **Parameter** — a number describing the population (e.g., true population mean μ) — usually unknown
- **Statistic** — a number computed from a sample (e.g., sample mean x̄) — what we calculate
- **Standard error** — the standard deviation of a sample statistic across many samples

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# Create a "population" — imagine measuring daily spending of all customers
# Use a right-skewed distribution (realistic for spending)
rng = np.random.default_rng(42)
population = rng.exponential(scale=800, size=100_000)   # skewed, mean ≈ 800

pop_mean = population.mean()
pop_std  = population.std()
pop_med  = np.median(population)

print("=== POPULATION (100,000 customers) ===")
print(f"Mean   : ₹{pop_mean:.2f}")
print(f"Median : ₹{pop_med:.2f}")
print(f"Std    : ₹{pop_std:.2f}")
print(f"Shape  : right-skewed (mean >> median)")

In reality, you'd never have data on 100,000 customers. We created this "population" to demonstrate what sampling does to our estimates.

The spending distribution is heavily right-skewed — a few big spenders pull the mean up significantly above the median. This is typical for sales data, incomes, and many real-world measurements.

In [ ]:
# Take one sample of size n=50 and see what we get
n = 50
sample = rng.choice(population, size=n, replace=False)

sample_mean = sample.mean()
sample_std  = sample.std(ddof=1)   # ddof=1 for unbiased sample std

print(f"=== ONE SAMPLE (n={n}) ===")
print(f"Sample mean : ₹{sample_mean:.2f}  (population: ₹{pop_mean:.2f})")
print(f"Sample std  : ₹{sample_std:.2f}  (population: ₹{pop_std:.2f})")
print(f"Difference  : ₹{abs(sample_mean - pop_mean):.2f}")
print()
print("The sample mean is an estimate — it won't be exactly right.")
print("But how far off is it, on average?")

This is the fundamental problem of statistics: a single sample gives you one estimate, and you have no way of knowing how close it is to the true population value.

The Central Limit Theorem gives you the answer.

## 2) The Central Limit Theorem

**The CLT states:**

> Take repeated samples of size n from **any** population.  
> The distribution of the **sample means** will be approximately normal,  
> regardless of the shape of the original population,  
> as long as n is large enough (typically n ≥ 30).

And the distribution of sample means has:
- **Mean** = μ (same as the population mean)
- **Standard deviation** = σ / √n  (called the **standard error**)

This is remarkable. The population can be skewed, uniform, bimodal — it doesn't matter. Average enough values together, and the result follows a bell curve.

In [ ]:
# Demonstrate the CLT — take 5,000 samples, compute mean of each
n_samples   = 5_000
sample_size = 50

sample_means = [rng.choice(population, size=sample_size, replace=False).mean()
                for _ in range(n_samples)]
sample_means = np.array(sample_means)

# Theoretical values from CLT
theoretical_mean = pop_mean
theoretical_se   = pop_std / np.sqrt(sample_size)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: the original skewed population
axes[0].hist(population, bins=80, color="darkorange", edgecolor="none", alpha=0.8, density=True)
axes[0].axvline(pop_mean,   color="black",    linestyle="--", linewidth=2, label=f"Mean = ₹{pop_mean:.0f}")
axes[0].axvline(pop_med,    color="firebrick",linestyle="-",  linewidth=2, label=f"Median = ₹{pop_med:.0f}")
axes[0].set_title("Original Population(Right-Skewed Spending)", fontweight="bold")
axes[0].set_xlabel("Daily Spending (₹)")
axes[0].set_ylabel("Density")
axes[0].legend(fontsize=9)
axes[0].grid(linestyle="--", alpha=0.4)

# Middle: distribution of sample means
axes[1].hist(sample_means, bins=60, color="steelblue", edgecolor="none", alpha=0.8, density=True)
x_range = np.linspace(sample_means.min(), sample_means.max(), 300)
axes[1].plot(x_range, stats.norm.pdf(x_range, theoretical_mean, theoretical_se),
             color="firebrick", linewidth=2.5, label="Theoretical Normal (CLT)")
axes[1].axvline(sample_means.mean(), color="black", linestyle="--", linewidth=2,
                label=f"Mean of means ≈ ₹{sample_means.mean():.0f}")
axes[1].set_title(f"Distribution of 5,000 Sample Means(n={sample_size} each)", fontweight="bold")
axes[1].set_xlabel("Sample Mean (₹)")
axes[1].legend(fontsize=9)
axes[1].grid(linestyle="--", alpha=0.4)

# Right: show how sample size affects the spread
colours = ["#EF9A9A","#EF5350","#C62828"]
for size, colour in zip([10, 30, 100], colours):
    means = [rng.choice(population, size=size, replace=False).mean() for _ in range(2000)]
    se = pop_std / np.sqrt(size)
    axes[2].hist(means, bins=40, density=True, alpha=0.5, color=colour, edgecolor="none",
                label=f"n={size}  (SE=₹{se:.0f})")
axes[2].set_title("Larger Sample → Narrower Distribution", fontweight="bold")
axes[2].set_xlabel("Sample Mean (₹)")
axes[2].legend(fontsize=9)
axes[2].grid(linestyle="--", alpha=0.4)

fig.suptitle("The Central Limit Theorem in Action", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/home/claude/06_stats/clt_demo.png", dpi=120)
plt.close()
print("CLT demo plot saved.")
print()
print(f"Population mean       : ₹{pop_mean:.2f}")
print(f"Mean of sample means  : ₹{sample_means.mean():.2f}  (should ≈ population mean)")
print(f"Std of sample means   : ₹{sample_means.std():.2f}  (should ≈ σ/√n = {theoretical_se:.2f})")

**What the three charts show:**

1. **Population** — heavily right-skewed. Not normal at all.
2. **Distribution of sample means** — nearly perfect bell curve despite the skewed population. The CLT worked.
3. **Effect of sample size** — larger n → narrower bell → more precise estimates

The key insight from chart 3: **the standard error (σ/√n) shrinks as sample size grows**. To halve the standard error, you need 4× the sample size (because of the √n).

## 3) Standard Error — How Precise is Your Estimate?

The standard error (SE) tells you how much the sample mean typically varies from the true population mean.

```
SE = σ / √n
```

**In practice:** You usually don't know σ (the population standard deviation). You estimate it using the sample standard deviation s:

```
SE ≈ s / √n
```

In [ ]:
# Standard error changes with sample size
pop_std_actual = pop_std   # we "know" this only because we built the population

print("Standard Error for Different Sample Sizes:")
print("-" * 50)
print(f"{'Sample Size':>12} {'SE (₹)':>12} {'95% margin of error':>22}")
print("-" * 50)

for n in [10, 25, 50, 100, 200, 500, 1000]:
    se = pop_std_actual / np.sqrt(n)
    margin = 1.96 * se    # 95% confidence uses z ≈ 1.96
    print(f"{n:>12} {se:>12.1f} ±{margin:>20.1f}")

print()
print(f"Population std: ₹{pop_std_actual:.1f}")
print("Margin of error = 1.96 × SE (for 95% confidence)")

**Reading the table:** With n=50, the standard error is about ₹113. This means sample means will typically be within ±₹221 of the true mean (95% of the time).

With n=1000, the margin shrinks to ±₹50.

This explains why polling organisations use samples of ~1,000–2,000: beyond that, the improvement in precision (√n shrinkage) slows down and costs money to gain little accuracy.

## 4) Confidence Intervals — What Range Probably Contains the Truth?

A **confidence interval (CI)** gives a range of plausible values for the population parameter.

A 95% CI means: if we repeated this sampling process 100 times and built a CI each time, **approximately 95 of those intervals would contain the true population mean**.

It does NOT mean "there's a 95% chance the population mean is in this interval." The population mean is a fixed (unknown) number — it's either in the interval or it isn't.

**Formula for a 95% CI (when σ is known):**

```
CI = x̄ ± 1.96 × (σ / √n)
```

**When σ is unknown** (the usual case), use the t-distribution:

```
CI = x̄ ± t*(α/2, df) × (s / √n)
```

In [ ]:
# Build a 95% confidence interval from one sample
sample = rng.choice(population, size=50, replace=False)
x_bar  = sample.mean()
s      = sample.std(ddof=1)
n      = len(sample)
se     = s / np.sqrt(n)

# t-distribution (df = n-1) for unknown population std
t_star = stats.t.ppf(0.975, df=n - 1)   # two-tailed 95%

lower = x_bar - t_star * se
upper = x_bar + t_star * se

print(f"Sample size      : {n}")
print(f"Sample mean (x̄)  : ₹{x_bar:.2f}")
print(f"Sample std (s)   : ₹{s:.2f}")
print(f"Standard error   : ₹{se:.2f}")
print(f"t* (95%, df=49)  : {t_star:.4f}")
print()
print(f"95% Confidence Interval: ₹{lower:.2f} to ₹{upper:.2f}")
print()
print(f"True population mean    : ₹{pop_mean:.2f}  ← contained? {lower <= pop_mean <= upper}")

In [ ]:
# Demonstrate: if we build 50 CIs, roughly 95% should contain the true mean
n_intervals = 50
contained   = 0
fig, ax     = plt.subplots(figsize=(10, 8))

for i in range(n_intervals):
    samp = rng.choice(population, size=50, replace=False)
    xb   = samp.mean()
    s    = samp.std(ddof=1)
    se_  = s / np.sqrt(len(samp))
    t_   = stats.t.ppf(0.975, df=len(samp)-1)
    lo   = xb - t_ * se_
    hi   = xb + t_ * se_
    hit  = lo <= pop_mean <= hi
    if hit:
        contained += 1
    colour = "steelblue" if hit else "firebrick"
    ax.plot([lo, hi], [i, i], color=colour, linewidth=1.5)
    ax.plot(xb, i, "o", color=colour, markersize=4)

ax.axvline(pop_mean, color="black", linewidth=2.5, linestyle="--", label=f"True mean ₹{pop_mean:.0f}")
ax.set_title(f"50 Confidence Intervals — {contained} contain the true mean  ({contained*2:.0f}%)Blue = contains true mean, Red = misses",
             fontweight="bold")
ax.set_xlabel("Daily Spending (₹)")
ax.set_ylabel("Sample #")
ax.legend()
ax.grid(linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig("/home/claude/06_stats/ci_demo.png", dpi=120)
plt.close()
print(f"CIs that contain true mean: {contained}/50 ({contained*2}%)")

The red intervals missed. They're not wrong — they were built correctly from their sample. But those samples happened to be atypical. About 5% of properly constructed 95% CIs will miss the true value, simply by chance.

This is what "95% confidence" means. Not that any single interval is right 95% of the time — it's a statement about the **procedure** producing intervals that work 95% of the time.

## 5) Sampling Methods — How You Sample Matters

The CLT and confidence intervals assume **random sampling**. The results don't hold if your sample is biased.

**Simple Random Sampling** — every member of the population has an equal chance of being selected. The gold standard.

**Stratified Sampling** — divide the population into groups (strata) and sample from each. Better coverage when groups differ.

**Cluster Sampling** — divide into clusters, randomly select clusters, sample all members in chosen clusters. Practical for geographically spread populations.

**Convenience Sampling** — take whoever is easy to reach. Cheap but biased. Should never be used for statistical inference.

In [ ]:
# Demonstrate bias from non-random sampling
# Suppose high spenders are more likely to respond to a survey (response bias)

# Probability of participating increases with spending
prob_respond = np.clip(population / population.max() * 0.5 + 0.1, 0.1, 0.8)
mask = rng.random(len(population)) < prob_respond

biased_sample   = population[mask]
random_sample   = rng.choice(population, size=500, replace=False)

print("=== Sampling Bias Demonstration ===")
print()
print(f"True population mean     : ₹{pop_mean:.2f}")
print()
print(f"Random sample mean       : ₹{random_sample.mean():.2f}  "
      f"(error: ₹{abs(random_sample.mean()-pop_mean):.2f})")
print(f"Biased sample mean       : ₹{biased_sample.mean():.2f}  "
      f"(error: ₹{abs(biased_sample.mean()-pop_mean):.2f})")
print()
print(f"Biased sample size       : {len(biased_sample)}")
print("High spenders are over-represented → mean is inflated.")

The biased sample produces a dramatically wrong estimate — not because the sample is small, but because the sampling process was unfair.

**More data doesn't fix a biased sampling method.** You can survey a million people and still be wrong if high spenders self-select to respond. This is why polling methodology matters more than poll sample size.

## Summary

| Concept | Key Formula | What it tells you |
|---|---|---|
| Standard Error | `SE = σ/√n` | How much sample means vary |
| CLT | Sample means → Normal(μ, σ/√n) | Why averages are predictable |
| 95% CI | `x̄ ± 1.96 × SE` (z) or `x̄ ± t* × SE` (t) | Plausible range for population mean |
| t vs z | Use t when σ unknown (always) | t has heavier tails, more conservative |

**The big ideas:**
- The sample mean is a random variable — it varies from sample to sample
- CLT says: averages follow a normal distribution regardless of population shape
- Larger samples → smaller SE → narrower CIs → more precise estimates
- But **bias can't be fixed by bigger samples** — only better sampling methods help

Next up: **Hypothesis Testing Part I** — formally testing whether an observed difference is real or just random chance.

## Practice Questions 1

1. A factory produces bolts with a mean length of 10 cm and std of 0.5 cm. If you take a sample of 64 bolts:
   - What is the standard error of the sample mean?
   - What is the probability the sample mean is within 0.1 cm of the true mean?
   - Use `stats.norm.cdf()` to compute this.

2. Using the spending population from this notebook (mean ≈ ₹800, std ≈ ₹800):
   - Take a sample of size n=100 and compute a 95% confidence interval.
   - Does it contain the true population mean?
   - Repeat 5 times. How many of your 5 CIs contained the true mean?

## Practice Questions 2

1. A newspaper claims that the average monthly mobile data usage in India is 18 GB. You survey 40 users and find a mean of 20.5 GB with std of 7 GB.
   - Build a 95% confidence interval for the true mean.
   - Does 18 GB fall inside your interval?

2. Why does the CLT matter for machine learning?
   - Think about what a model's validation score on a test set represents
   - How would the standard error concept apply to understanding model performance?
   - (Write your answer in a new cell as markdown or a print statement.)